# Tutorial 7 — Image-text representations: CLIP and SigLIP

[Lecture notes](../lectures/07_contrastive.md) · [Slides](../slides/07_contrastive.pptx) · [Exercise sheet](../exercises/07_contrastive.md)

**40 minutes, CPU, NumPy only.** You will train a small image-to-text alignment model on generated pixel arrays, test it on a colour-position combination it never saw, work one contrastive loss by hand, and reuse the frozen image encoder as a feature extractor. The images are 8×8 arrays and the "text encoder" is a hand-written vector per label. Nothing here measures CLIP or SigLIP.

| Minutes | Step |
|---|---|
| 0–5 | Build six images and their label vectors |
| 5–15 | Train with the symmetric contrastive loss |
| 15–20 | Test the held-out combination |
| 20–25 | A 3×3 loss by hand, softmax and sigmoid |
| 25–30 | Freeze the encoder and fit a logistic regression on its features |
| 30–40 | Your turn: two functions with tests |

![The six image-label combinations, with blue-right held out.](../data/paired-shapes.svg)

In [1]:
import sys
import numpy as np
rng = np.random.default_rng(42)
print("Python", sys.version.split()[0], "NumPy", np.__version__, "seed", 42)

Python 3.13.1 NumPy 2.1.3 seed 42


## 1. Generate paired image and label features

Each image is an 8×8 RGB array with one coloured block on the left or the right. Each label vector is a one-hot colour joined to a one-hot position, so "blue right" shares three numbers with "blue left". Blue-right is held out of training.

Before running: how many numbers does one flattened image hold?

In [2]:
colors=np.eye(3); names=["red","green","blue"]
images=[]; text=[]; labels=[]
for c in range(3):
    for side in range(2):
        im=np.zeros((8,8,3)); start=0 if side==0 else 4
        im[2:6,start:start+4]=colors[c]
        images.append(im)
        text.append(np.r_[colors[c],np.eye(2)[side]])
        labels.append(names[c]+(" left" if side==0 else " right"))
images=np.array(images); text=np.array(text)
X=images.reshape(6,-1); X/=np.maximum(np.linalg.norm(X,axis=1,keepdims=True),1e-12)
T=text/np.linalg.norm(text,axis=1,keepdims=True)
train=np.arange(5); test=np.array([5]); tau=.2
W=rng.normal(0,.05,(X.shape[1],T.shape[1]))
def softmax(z):
    z=z-z.max(axis=-1,keepdims=True); e=np.exp(z); return e/e.sum(axis=-1,keepdims=True)
print("image batch",images.shape,"labels",labels,"held out",labels[5])
print("cosine(text 'blue left', text 'blue right') =",round(float(T[4]@T[5]),2))

image batch (6, 8, 8, 3) labels ['red left', 'red right', 'green left', 'green right', 'blue left', 'blue right'] held out blue right
cosine(text 'blue left', text 'blue right') = 0.5


One image is 8 × 8 × 3 = 192 numbers. The image encoder is a single matrix `W` of shape 192 × 5. The two blue label vectors have cosine 0.5 because they share the colour and differ in the position. Keep that 0.5 in mind for section 3.

## 2. Train with the symmetric contrastive loss

The loop normalises the image embeddings, builds the 5×5 similarity matrix `S` with temperature 0.2, and averages the row-wise and column-wise cross-entropy. The gradient with respect to `S` is `(row softmax − I + column softmax − I) / 2N`.

Before running: a model that cannot tell five captions apart has loss log 5 = 1.609. Will the loss at step 0 be above or below that?

In [3]:
losses=[]
for step in range(201):
    Z=X[train]@W; norms=np.maximum(np.linalg.norm(Z,axis=1,keepdims=True),1e-12); U=Z/norms
    S=U@T[train].T/tau; row=softmax(S); col=softmax(S.T)
    loss=-.5*(np.log(np.diag(row)).mean()+np.log(np.diag(col)).mean())
    losses.append(float(loss))
    dS=(row-np.eye(5)+col.T-np.eye(5))/10
    dU=dS@T[train]/tau
    dZ=(dU-U*(dU*U).sum(axis=1,keepdims=True))/norms
    W-=.1*X[train].T@dZ
    if step%50==0: print(step,"loss",round(float(loss),4),"   log(5) =",round(float(np.log(5)),4))
assert losses[-1]<losses[0]

0 loss 2.5377    log(5) = 1.6094
50 loss 0.0879    log(5) = 1.6094
100 loss 0.0799    log(5) = 1.6094
150 loss 0.0774    log(5) = 1.6094
200 loss 0.0763    log(5) = 1.6094


The loss at step 0 is 2.54, above log 5. The weights start small, but the embeddings are normalised to length 1, so the starting cosines are random and the temperature multiplies them by 5. A randomly wrong model with a sharp softmax does worse than a uniform guess. The loss then levels off near 0.076. It cannot reach zero: a cosine is at most 1, so no logit exceeds 1/0.2 = 5, and captions that share a colour or a position have cosine 0.5 with each other.

## 3. Test the held-out combination

All six captions are candidates now. Before running: which caption will the blue-right image pick, and why?

In [4]:
Z=X@W; U=Z/np.maximum(np.linalg.norm(Z,axis=1,keepdims=True),1e-12)
S=U@T.T/tau
for i in range(6): print(labels[i],"->",labels[int(S[i].argmax())],"held-out" if i==5 else "train")
print("image-to-text top1",np.mean(S.argmax(axis=1)==np.arange(6)))
print("text-to-image top1",np.mean(S.argmax(axis=0)==np.arange(6)))
print("similarity matrix\n",np.round(S,2))
# Uniform logits must give log(N), regardless of temperature.
assert np.isclose(-np.log(softmax(np.zeros((3,3)))[0,0]),np.log(3))

red left -> red left train
red right -> red right train
green left -> green left train
green right -> green right train
blue left -> blue left train
blue right -> blue right held-out
image-to-text top1 1.0
text-to-image top1 0.8333333333333334
similarity matrix
 [[ 3.37  0.34 -0.09 -3.12  0.04 -2.99]
 [ 0.    3.59 -3.47  0.12 -2.04  1.54]
 [-0.21 -3.23  3.27  0.25  0.05 -2.97]
 [-3.48  0.12 -0.06  3.54 -2.28  1.32]
 [-0.51 -1.8  -0.54 -1.83  3.65  2.36]
 [-4.12 -0.07 -2.51  1.53 -2.38  1.67]]


Read the last row and the last column. The blue-right image picks its own caption, but by 1.67 against 1.53, a margin of 0.14 where every training image wins by more than 1. The caption "blue right" retrieves the blue-left image (2.36 against 1.67), which is the text-to-image miss. The right-hand pixels of the blue channel were zero in every training image, so their weights in `W` never moved from their random start, and the blue-right embedding is a random direction. Its correct top-1 is luck: change the seed and count how often it survives. The text side can compose "blue" and "right". The image side has never seen those pixels.

## 4. A 3×3 loss by hand

Take three images and three captions with these logits (similarity already divided by the temperature). Rows are images, columns are captions, and the matched pairs sit on the diagonal.

Before running: which image is matched to the wrong caption? Which direction of the loss will be larger?

In [5]:
S3=np.array([[3.,1.,0.],[1.,2.,3.],[0.,1.,4.]])
row_loss=-np.log(np.diag(softmax(S3)))        # image -> text: softmax along each row
col_loss=-np.log(np.diag(softmax(S3.T)))      # text -> image: softmax along each column
print("image-to-text terms",np.round(row_loss,4),"mean",round(float(row_loss.mean()),4))
print("text-to-image terms",np.round(col_loss,4),"mean",round(float(col_loss.mean()),4))
print("symmetric loss",round(float((row_loss.mean()+col_loss.mean())/2),4))
def log_sigmoid(x): return -np.log1p(np.exp(-x))
signs=2*np.eye(3)-1                             # +1 on the diagonal, -1 elsewhere
for b in [0.,-2.]:
    print("sigmoid loss with bias",b,"=",round(float(-log_sigmoid(signs*(S3+b)).sum()/3),4))
assert np.isclose(row_loss[1],np.log(np.e+np.e**2+np.e**3)-2)

image-to-text terms [0.1698 1.4076 0.0659] mean 0.5478
text-to-image terms [0.1698 0.5514 0.3266] mean 0.3493
symmetric loss 0.4485
sigmoid loss with bias 0.0 = 2.8561
sigmoid loss with bias -2.0 = 1.2134


Image 2 scores 3 against caption 3 and only 2 against its own caption, so its row term is the largest number printed. Caption 2 still ranks image 2 first, so the text-to-image mean is smaller. Check one term with a calculator: row 2 is log(e + e² + e³) − 2.

The sigmoid loss treats the nine cells as nine yes/no questions. With bias 0 the six negative cells dominate. A negative bias lowers every logit and the loss falls, which is why SigLIP learns a bias and starts it at −10.

## 5. Freeze the encoder, train a small classifier

Three of the four capstone tasks have images. The cheapest image baseline keeps a pretrained encoder frozen, stores one feature vector per image, and fits a logistic regression on top. Here the "pretrained encoder" is the `W` you trained above. The new task is one that contrastive training never named: is the block on the right? The images get pixel noise so that the task is not trivial.

Before running: the encoder never saw blue-right. Will a classifier trained on the other five combinations get the position of blue-right images correct?

In [6]:
def encode(flat):
    z=flat@W; return z/np.maximum(np.linalg.norm(z,axis=1,keepdims=True),1e-12)

def fit_logreg(F,y,steps=500,lr=1.0):
    w=np.zeros(F.shape[1]); b=0.0
    for _ in range(steps):
        p=1/(1+np.exp(-(F@w+b))); w-=lr*F.T@(p-y)/len(y); b-=lr*(p-y).mean()
    return w,b

combo=np.repeat(np.arange(6),60)
noisy=X[combo]+rng.normal(0,.05,(len(combo),X.shape[1]))
is_right=(combo%2).astype(float)
seen=combo!=5; half=rng.random(len(combo))<.5
fit=seen&half; heldout_rows=seen&~half
for name,F in [("frozen encoder features (5 numbers)",encode(noisy)),("raw pixels (192 numbers)",noisy)]:
    w,b=fit_logreg(F[fit],is_right[fit])
    acc=lambda rows:(((F[rows]@w+b)>0)==is_right[rows]).mean()
    print(f"{name}: seen combinations {acc(heldout_rows):.3f}   blue-right {acc(~seen):.3f}")
    if F.shape[1]==5: probe_seen=acc(heldout_rows)
assert probe_seen>.9

frozen encoder features (5 numbers): seen combinations 1.000   blue-right 0.750
raw pixels (192 numbers): seen combinations 1.000   blue-right 0.033


On combinations the encoder was trained on, five frozen features do as well as 192 pixels. On blue-right the pixel classifier scores 0.033: it has no weight on pixels it never saw lit, so it falls back on its bias and says "left". The frozen features score 0.750, but section 3 showed that the encoder's blue-right direction is random, so do not read 0.750 as skill. An accuracy of 1.000 on familiar images told you nothing about either number. In your capstone the same pattern is `features = encoder(images)` computed once and saved, then `LogisticRegression().fit(features, labels)`.

## 6. Your turn

Replace each `raise` with your code, then run the test cell under it. Do not edit the tests.

**A. Symmetric contrastive loss from a similarity matrix.** `S[i, j]` is the logit between image `i` and caption `j`. Matched pairs are on the diagonal. Return the mean of the image-to-text and text-to-image cross-entropies.

In [ ]:
# YOUR TURN
def contrastive_loss(S):
    """Symmetric contrastive loss of a square logit matrix with matched pairs on the diagonal."""
    raise NotImplementedError("your turn")

In [8]:
assert np.isclose(contrastive_loss(np.zeros((3,3))),np.log(3))
assert np.isclose(contrastive_loss(S3),(row_loss.mean()+col_loss.mean())/2)
assert np.isclose(contrastive_loss(S3),contrastive_loss(S3.T))          # swapping the two directions changes nothing
assert contrastive_loss(100*np.eye(4))<1e-6
print("A passed: loss of the worked matrix =",round(float(contrastive_loss(S3)),4))

A passed: loss of the worked matrix = 0.4485


**B. Recall@k in both directions.** For image `i` the relevant caption is caption `i`. Image-to-text recall@k is the share of rows whose diagonal entry is among the `k` largest in the row. Text-to-image recall@k does the same down the columns. Return both numbers.

In [ ]:
# YOUR TURN
def recall_at_k(S,k):
    """Return (image-to-text recall@k, text-to-image recall@k) for matched pairs on the diagonal."""
    raise NotImplementedError("your turn")

In [10]:
r1=recall_at_k(S3,1); r2=recall_at_k(S3,2)
assert np.allclose(r1,(2/3,1.0)) and np.allclose(r2,(1.0,1.0))
# A larger gallery: 200 paired unit vectors. The "text tower" adds noise of size sigma to each caption embedding.
N,d=200,32
img=rng.normal(size=(N,d)); img/=np.linalg.norm(img,axis=1,keepdims=True)
print("sigma  i2t R@1  t2i R@1  i2t R@5  t2i R@5")
for sigma in [0.0,0.1,0.2,0.3,0.4]:
    txt=img+sigma*rng.normal(size=(N,d)); txt/=np.linalg.norm(txt,axis=1,keepdims=True)
    a1,b1=recall_at_k(img@txt.T,1); a5,b5=recall_at_k(img@txt.T,5)
    print(f"{sigma:5.1f}  {a1:7.3f}  {b1:7.3f}  {a5:7.3f}  {b5:7.3f}")
    if sigma==0: assert a1==b1==1.0
print("B passed: worked matrix recall@1 =",np.round(r1,3))

sigma  i2t R@1  t2i R@1  i2t R@5  t2i R@5
  0.0    1.000    1.000    1.000    1.000
  0.1    1.000    1.000    1.000    1.000
  0.2    0.985    0.970    1.000    1.000
  0.3    0.655    0.630    0.895    0.910
  0.4    0.365    0.380    0.680    0.690
B passed: worked matrix recall@1 = [0.667 1.   ]


The table is a stand-in for a text tower that embeds one language less precisely than another: the images are the same and only the caption vectors get noisier. Recall@5 hides damage that recall@1 shows. A random ranker would score 1/200 at k = 1 and 5/200 at k = 5; report that floor beside your numbers.

## Discussion

1. Why normalise embeddings before comparing them?
2. Two captions in a batch are both correct for one image. What does the diagonal-only loss do, and what would you change?
3. A zero-shot classifier returns 0.9 for "a photo of a goat". Is that a probability that the image shows a goat?

## From the toy to your task

Task 12 subtask 3 ships a baseline of this shape: frozen CLIP ViT-B/32 image features and a logistic regression, with a within-painting correlation of 0.45. Task 9 subtask 2 labels news images with frames, and a frozen encoder plus a linear classifier is the first baseline to run there too. The optional script `course/extensions/hf_vision.py --mode clip` loads a pretrained CLIP; the course has not run it for you, so see the [extension guide](../extensions/README.md) first.

Submit this notebook with both tests passing and a 150–250-word note: what the held-out row shows, one number that supports it, and one way the toy differs from a pretrained encoder.